<a href="https://colab.research.google.com/github/Eva360563/Deep_Natural_language/blob/main/2026_27_Practice_2_Word_and_Sentence_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistant:** Ali Yassine & Lorenzo Vaiani

**Credits:** Moreno La Quatra

**Practice 2:** Word and Sentence Embeddings

## Word Embedding

![](https://qph.fs.quoracdn.net/main-qimg-3e812fd164a08f5e4f195000fecf988f)


**Key takeaways** from lessons and in-class practices:
- Word embeddings are able to map words into a semantic-aware vector space.
- There are multiple architectures for the generation of word embeddings.
- Each architecture has its advantages and disadvantages.
- Word embedding evaluation could be intrinsic (intermediate tasks) or extrinsic (downstream task).
- It is possible to use pre-trained word embedding models or use large amount of text to train it from scratch.
- The use of pre-trained word embedding models is a common practice in NLP and removes the need of training a word embedding model from scratch (that could be very time consuming and computationally expensive).

### **Question 1**

Train a new Word2Vec model using gensim with the text8 corpus available in the Python package ([reference](https://radimrehurek.com/gensim/downloader.html)). Compute the training time for the model and store it for subsequent steps.

**Hint:** you can use the following code to load the text8 corpus:

```python
import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus
import time
path = api.load("text8", return_path=True)
dataset = Text8Corpus(path)
```

In [1]:
! pip install --upgrade gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 28.3 MB/s eta 0:00:00


In [2]:
# your code here
import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus
import time
path = api.load("text8", return_path=True)
dataset = Text8Corpus(path)
start_time= time.time()
model= Word2Vec(sentences=dataset, vector_size=100, window= 5, min_count=5, workers=4)
end_time= time.time()
training_time= end_time-start_time
print(f"Total training time: {training_time:.2f} seconds")

[==================================================] 100.0% 31.6/31.6MB downloaded
Total training time: 171.45 seconds


### **Question 2**
Perform **intrinsic** evaluation of the model for the task of word analogy by exploiting the data collection available [here](https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv).

1. read CSV file
2. group analogy entries by type (column: `type`)
3. for each type of entry (**in the lab, just set type="family"** to reduce the required time) use the first 3 word vectors to compute the fourth
    - Entry: `Athens,Greece,Baghdad,Iraq`
    - `v(Greece) - v(Athens) + v(Baghdad) = res_v`
    - Get the most similar vectors to `res_v`
    - Compute in how many cases the correct word is among the top k (if `v[Iraq]` is among the k most similar words) with `k = 1, 3, 5, 10`

$top(k) = \dfrac{\sum_{i=1}^{N} f(i)}{|E|}$

where $f(i) = 1$ if the target word is among the top k and $f(i) = 0$ otherwise.

$|E|$ is the total number of entries for the considered type.

**Notes:**
1. Try with the model trained on `text8`, is there any issue? If yes, how can you solve it?
2. Test the model trained on Google News available in gensim.

In [3]:
%%capture
! wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv
! pip install --upgrade pandas

In [4]:
# Executing this cell could take ~5 minutes
import gensim.downloader
w2v_google_news_model = gensim.downloader.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [6]:
# your code here
import pandas as pd
df= pd.read_csv('google_analogies.csv')
def evaluate_analogies(model, dataframe, analogy_type= 'family', lowercase= False):
  subset= dataframe[dataframe['type']== analogy_type]
  total_entries_E= len(subset)
  top_k_hits= {1: 0, 3:0, 5:0, 10: 0}
  vec_model= model.wv if hasattr(model, 'wv') else model
  vocab = vec_model.key_to_index
  for _, row in subset.iterrows():
    w1, w2 =row['word1'], row['word2']
    w3, w4 =row['word3'], row['target']
    if lowercase:
      w1, w2, w3, w4 = w1.lower(), w2.lower(), w3.lower(), w4.lower()
    if w1 not in vocab or w2 not in vocab or w3 not in vocab:
      continue
    try:
      # We fetch top 10 to cover our max k=10 requirement
      predictions = vec_model.most_similar(positive=[w2, w3], negative=[w1], topn=10)
      predicted_words = [word for word, similarity in predictions]

        # Check if the correct target word (w4) is in the top-k predictions
      for k in top_k_hits.keys():
        if w4 in predicted_words[:k]:
          top_k_hits[k] += 1

    except Exception as e:
      continue
  print(f"--- Evaluation for type: '{analogy_type}' ---")
  print(f"Total entries (|E|): {total_entries_E}")
  for k in [1, 3, 5, 10]:
        accuracy = top_k_hits[k] / total_entries_E
        print(f"Top-{k} Accuracy: {accuracy:.4f}")
  print("\n")

# --- Run the evaluations ---

# Note: Assuming 'model' is the Word2Vec model trained on text8 from the previous step
print("Evaluating Text8 Model (with lowercase correction):")
evaluate_analogies(model, df, analogy_type="family", lowercase=True)

# Evaluating the Google News model
print("Evaluating Google News Model:")
evaluate_analogies(w2v_google_news_model, df, analogy_type="family", lowercase=False)

Evaluating Text8 Model (with lowercase correction):
--- Evaluation for type: 'family' ---
Total entries (|E|): 506
Top-1 Accuracy: 0.4862
Top-3 Accuracy: 0.5889
Top-5 Accuracy: 0.6265
Top-10 Accuracy: 0.6700


Evaluating Google News Model:
--- Evaluation for type: 'family' ---
Total entries (|E|): 506
Top-1 Accuracy: 0.8458
Top-3 Accuracy: 0.9229
Top-5 Accuracy: 0.9526
Top-10 Accuracy: 0.9743




### **Question 3**

Train a new FastText model using gensim with text8 corpus available in the Python package ([reference](https://radimrehurek.com/gensim/downloader.html)). Compute the training time for the model and store it for subsequent steps.

- Is there any significant difference in training time if compared with Word2Vec training?

In [7]:
# your code here
from gensim.models import FastText
import time

# Assumiamo che 'dataset' sia già disponibile dall'Esercizio 1
# dataset = Text8Corpus(path)

print("Inizio l'addestramento del modello FastText (potrebbe volerci un po')...")
start_time_ft = time.time()

# Addestramento del modello FastText
# Manteniamo gli stessi parametri di Word2Vec per un confronto alla pari
ft_model = FastText(sentences=dataset, vector_size=100, window=5, min_count=5, workers=4)

end_time_ft = time.time()
training_time_ft = end_time_ft - start_time_ft

print("Modello FastText addestrato con successo.")
print(f"Tempo di addestramento FastText: {training_time_ft:.2f} secondi")

# Se hai ancora in memoria la variabile 'training_time' del Word2Vec precedente:
# print(f"Tempo Word2Vec precedente: {training_time:.2f} secondi")

Inizio l'addestramento del modello FastText (potrebbe volerci un po')...
Modello FastText addestrato con successo.
Tempo di addestramento FastText: 682.47 secondi


### **Question 4**
Provide the same evaluation done in Question 2 for the FastText model. In this case, you can use the same type of analogy (family) and the same k values.

**Notes:**
- Try with the model trained on `text8`, is there any issue? What does it mean?
- Test the model trained on Wikipedia+News available in gensim.

In [8]:
# Executing this cell could take ~5 minutes
import gensim.downloader
ft_wiki_news_model = gensim.downloader.load('fasttext-wiki-news-subwords-300')

[==================================================] 100.0% 958.5/958.4MB downloaded


In [9]:
# your code here
import gensim.downloader as api
import pandas as pd

def evaluate_fasttext_analogies(model, dataframe, analogy_type='family', lowercase=False):
    subset = dataframe[dataframe['type'] == analogy_type]
    total_entries_E = len(subset)

    if total_entries_E == 0:
        return

    top_k_hits = {1: 0, 3: 0, 5: 0, 10: 0}
    vec_model = model.wv if hasattr(model, 'wv') else model

    for _, row in subset.iterrows():
        w1, w2 = str(row['word1']), str(row['word2'])
        w3, w4 = str(row['word3']), str(row['target'])

        if lowercase:
            w1, w2, w3, w4 = w1.lower(), w2.lower(), w3.lower(), w4.lower()

        try:
            # FastText calcola il vettore al volo anche se la parola non è nel vocabolario!
            predictions = vec_model.most_similar(positive=[w2, w3], negative=[w1], topn=10)
            predicted_words = [word for word, similarity in predictions]

            for k in top_k_hits.keys():
                if w4 in predicted_words[:k]:
                    top_k_hits[k] += 1

        except Exception as e:
            continue

    print(f"\n--- Valutazione FastText per il tipo: '{analogy_type}' ---")
    print(f"Totale entry (|E|): {total_entries_E}")
    for k in [1, 3, 5, 10]:
        accuracy = top_k_hits[k] / total_entries_E
        print(f"Top-{k} Accuracy: {accuracy:.4f}")
    print("-" * 40)

# 1. Valutazione del modello FastText addestrato su text8
print("Valutazione FastText Text8 (con conversione minuscolo):")
evaluate_fasttext_analogies(ft_model, df, analogy_type="family", lowercase=True)

# 2. Valutazione del modello FastText pre-addestrato
print("Valutazione FastText Wiki+News (originale):")
evaluate_fasttext_analogies(ft_wiki_news_model, df, analogy_type="family", lowercase=False)

Valutazione FastText Text8 (con conversione minuscolo):

--- Valutazione FastText per il tipo: 'family' ---
Totale entry (|E|): 506
Top-1 Accuracy: 0.2668
Top-3 Accuracy: 0.4486
Top-5 Accuracy: 0.4901
Top-10 Accuracy: 0.5514
----------------------------------------
Valutazione FastText Wiki+News (originale):

--- Valutazione FastText per il tipo: 'family' ---
Totale entry (|E|): 506
Top-1 Accuracy: 0.8498
Top-3 Accuracy: 0.9506
Top-5 Accuracy: 0.9605
Top-10 Accuracy: 0.9842
----------------------------------------


### **Question 5** (optional)

Provide a complete evaluation of the best performing models (Word2Vec and FastText) by leveraging the complete dataset of analogy entries. In this case, you should use all the analogy types and all you can use the same k values provided in Question 2.

In [ ]:
# your code here

## Sentence Embeddings

Sentence embeddings are a way to represent a sentence in a vector space. The vector space is usually learned from a large corpus of text. They are used in many NLP tasks, such as text classification, text similarity, and question answering. In this practice, we will use and interact both with Doc2Vec and InferSent models.

**Key takeaways** from lessons and in-class practices:
- Doc2Vec is an extension of the Word2Vec framework.
- It incorporates Document ID to obtain a more accurate representation of a document/paragraph.
- Training document vectors are pre-computed, however you can infer vectors for new documents.
- InferSent exploit a deep learning architecture to supervisedly learn sentence representations.
- InferSent vectors could exploit both Word2Vec or FastText as word embedding models.

### **Question 6**

Train a novel Doc2Vec model using the [APIs provided by gensim](https://radimrehurek.com/gensim/models/doc2vec.html) with text8 corpus.

- Which is the training time for the model? Is it comparable with Word2Vec and FastText training time?

**Note:** Store the model to a file for subsequent steps.

In [10]:
# your code here
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.models.word2vec import Text8Corpus
import gensim.downloader as api
import time

# 1. Carichiamo text8 (se non già presente in memoria)
path = api.load("text8", return_path=True)
dataset = Text8Corpus(path)

# 2. Prepariamo i TaggedDocument assegnando un tag univoco a ciascun blocco di testo
print("Preparazione dei TaggedDocument...")
tagged_docs = [TaggedDocument(words=words, tags=[str(i)]) for i, words in enumerate(dataset)]
print(f"Numero totale di documenti creati: {len(tagged_docs)}")

# 3. Addestramento del modello Doc2Vec
print("\nInizio addestramento Doc2Vec...")
start_time_d2v = time.time()

# Manteniamo parametri coerenti con Word2Vec e FastText (epochs=5 come in Word2Vec)
d2v_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=5
)

end_time_d2v = time.time()
training_time_d2v = end_time_d2v - start_time_d2v

print("Addestramento Doc2Vec completato!")
print(f"Tempo di addestramento Doc2Vec: {training_time_d2v:.2f} secondi")

# 4. Salvataggio del modello su file richiesto dalla nota
d2v_model.save("doc2vec_text8.model")
print("Modello salvato su file: 'doc2vec_text8.model'")

Preparazione dei TaggedDocument...
Numero totale di documenti creati: 1701

Inizio addestramento Doc2Vec...
Addestramento Doc2Vec completato!
Tempo di addestramento Doc2Vec: 154.06 secondi
Modello salvato su file: 'doc2vec_text8.model'


### **Question 7 (Doc2Vec qualitative evaluation)**
Perform some **qualitative** experiments by computing the cosine similarities between sentences composed by yourself.
For example, you can use the following sentences:

```python
s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"
```

Please try to interact with the model by providing different sentences and check the results. Is the model able to capture the semantic meaning of the sentences? Are you satisfied with the results?

In [11]:
# your code here
import numpy as np
from gensim.models.doc2vec import Doc2Vec
import gensim.utils

# 1. Caricamento del modello salvato nello step precedente
model = Doc2Vec.load("doc2vec_text8.model")

# 2. Definizione delle frasi da testare
s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"
s5 = "A smartphone is a mobile electronic gadget" # Frase extra per test semantico

# Funzione helper per tokenizzare e calcolare il vettore del documento
def get_sentence_vector(sentence, d2v_model):
    # simple_preprocess esegue il lowercase e rimuove la punteggiatura
    tokens = gensim.utils.simple_preprocess(sentence)
    # infer_vector calcola le coordinate della nuova frase nello spazio esistente
    return d2v_model.infer_vector(tokens)

# Generazione dei vettori
v1 = get_sentence_vector(s1, model)
v2 = get_sentence_vector(s2, model)
v3 = get_sentence_vector(s3, model)
v4 = get_sentence_vector(s4, model)
v5 = get_sentence_vector(s5, model)

# Funzione per calcolare la Cosine Similarity (da -1 a 1, dove 1 è identico)
def cosine_similarity(vec_a, vec_b):
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b))

print("=== Valutazione Qualitativa Doc2Vec ===")
print(f"s1 vs s2 (Stessa struttura, soggetti diversi): {cosine_similarity(v1, v2):.4f}")
print(f"s1 vs s3 (Stesso macro-argomento US): {cosine_similarity(v1, v3):.4f}")
print(f"s1 vs s4 (Argomenti totalmente slegati): {cosine_similarity(v1, v4):.4f}")
print(f"s4 vs s5 (Stesso significato, parole diverse): {cosine_similarity(v4, v5):.4f}")

=== Valutazione Qualitativa Doc2Vec ===
s1 vs s2 (Stessa struttura, soggetti diversi): 0.4492
s1 vs s3 (Stesso macro-argomento US): 0.4482
s1 vs s4 (Argomenti totalmente slegati): 0.5044
s4 vs s5 (Stesso significato, parole diverse): 0.3138


### **Question 8**

Load the InferSent model provided by Facebook Research ([reference](https://github.com/facebookresearch/InferSent)) and perform the same qualitative evaluation done in Question 7. In this case, you can use the InferSent pretrained model (v2) - [reference](https://github.com/facebookresearch/InferSent).

Try to find some sentences for which InferSent is able to capture the semantic meaning of the sentences as opposed to Doc2Vec. Are you satisfied with the results? Which model is able to better capture the semantic meaning of the sentences? What can be the reason for this?

**Note:**
Please find below the code to download the InferSent model.

In [12]:
%%capture
# InferSent download required files

! mkdir fastText
! curl -Lo fastText/crawl-300d-2M.vec.zip https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip
! unzip fastText/crawl-300d-2M.vec.zip -d fastText/
! mkdir encoder
! curl -Lo encoder/infersent2.pkl https://dl.fbaipublicfiles.com/infersent/infersent2.pkl
! git clone https://github.com/facebookresearch/InferSent.git

In [3]:
from InferSent.models import InferSent
import torch
V = 2
MODEL_PATH = 'encoder/infersent%s.pkl' % V
params_model = {'bsize': 64, 'word_emb_dim': 300, 'enc_lstm_dim': 2048,
                'pool_type': 'max', 'dpout_model': 0.0, 'version': V}
infersent = InferSent(params_model)
infersent.load_state_dict(torch.load(MODEL_PATH))

W2V_PATH = 'fastText/crawl-300d-2M.vec'
infersent.set_w2v_path(W2V_PATH)

**Note:** Due to compatibility issues between newer NumPy versions and InferSent, you may encounter the following error when calling the `encode` method of the InferSent object:
> ValueError: setting an array element with a sequence...

If this occurs, you can fix it as follows:
- Restart the session
- Before loading the InferSent class, modify the `models.py` file in the InferSent folder by replacing line 207 with `sentences = np.array(sentences, dtype=object)[idx_sort]`

In [1]:
!sed -i 's/sentences = np.array(sentences)\[idx_sort\]/sentences = np.array(sentences, dtype=object)[idx_sort]/g' InferSent/models.py

In [2]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt') # Scarichiamo anche la versione base per sicurezza

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
# your code here
import numpy as np
from scipy.spatial.distance import cosine

# Definiamo le frasi
s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"

# FRASI TEST SEMANTICO (Stesso significato di s4, zero parole in comune)
s5 = "A smartphone is a mobile electronic gadget"

sentences = [s1, s2, s3, s4, s5]

# InferSent richiede di costruire il vocabolario sulle frasi che stiamo per analizzare
infersent.build_vocab(sentences, tokenize=True)

# Generazione dei vettori per tutte le frasi
embeddings = infersent.encode(sentences, tokenize=True)

# Funzione per calcolare la Cosine Similarity (usiamo 1 - cosine_distance)
def cos_sim(u, v):
    return 1 - cosine(u, v)

print("=== Valutazione Qualitativa InferSent ===")
print(f"s1 vs s2 (Stessa struttura): {cos_sim(embeddings[0], embeddings[1]):.4f}")
print(f"s1 vs s3 (Stesso macro-argomento): {cos_sim(embeddings[0], embeddings[2]):.4f}")
print(f"s1 vs s4 (Argomenti slegati): {cos_sim(embeddings[0], embeddings[3]):.4f}")
print(f"s4 vs s5 (STESSO SIGNIFICATO, PAROLE DIVERSE): {cos_sim(embeddings[3], embeddings[4]):.4f}")

Found 23(/23) words with w2v vectors
Vocab size : 23
=== Valutazione Qualitativa InferSent ===
s1 vs s2 (Stessa struttura): 0.8108
s1 vs s3 (Stesso macro-argomento): 0.5508
s1 vs s4 (Argomenti slegati): 0.0991
s4 vs s5 (STESSO SIGNIFICATO, PAROLE DIVERSE): 0.6542


### **Question 9** (Extrinsic Evaluation)

**Extrinsic** evaluation aims at measuring the performance of the word/sentence/paragraph embedding model when used in a downstream task. In this case, we will use the model to perform a text classification task.
We can use different configuration, training corpora or even different models to build a complete architecture for the task at hand.

For this practice we use the text classification dataset available [here](https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P2/news_headline_classification.csv) - [source: Kaggle](https://www.kaggle.com/rmisra/news-category-dataset). It contains news headlines and the corresponding category. The dataset is composed by 200846 divided into multiple categories (e.g. politics, business, sports, etc.).

**Note:** consider using just the first 10.000 headlines to reduce runtime during the lab. You can use the complete data collection at home to achieve better results.

Compute the accuracy of 3 classification models each one built with one of the models introduced in this practice:
- Word2Vec model pretrained on Google News corpus
- FastText model pretrained on Wikipedia+News corpus
- **[Optional]** Doc2Vec model pretrained on Text8 corpus
- **[Optional]** InferSent pretrained model (v2) - [reference](https://github.com/facebookresearch/InferSent)

The procedure to create a classification system is sketched below:
1. Choose a machine learning (multi-class) classifier (e.g., MLP)
2. Split the data collection in train/test (80%/20%)
3. Use text vectors obtained by pretrained model as input of the classifier
4. Measure the accuracy of the classification system
5. Repeat step 3-4 using different embedding models


**Note:** For word embedding models you must use an aggregation strategy to obtain a single vector for each sentence. You can use the average of the word vectors or the sum of the word vectors. In both cases, the output vector can be used as input of the classifier.

Report the performance of each classification pipeline. Which model has better performance? Why? Try to elaborate on the results.

In [5]:
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/news_headline_classification.csv

--2026-10-07 14:25:56--  https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/news_headline_classification.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15212143 (15M) [text/plain]
Saving to: ‘news_headline_classification.csv’

news_headline_class 100%[===================>]  14.51M  --.-KB/s    in 0.07s   

2026-10-07 14:25:57 (206 MB/s) - ‘news_headline_classification.csv’ saved [15212143/15212143]



In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import gensim.utils
import gensim.downloader as api

print("1. Ricaricamento Word2Vec (Google News)...")
w2v_google_news_model = api.load('word2vec-google-news-300')

print("2. Ricaricamento FastText (Wiki+News)...")
ft_wiki_news_model = api.load('fasttext-wiki-news-subwords-300')

print("\n3. Caricamento del dataset...")
df = pd.read_csv('news_headline_classification.csv')
df = df.head(10000)
print(f"Dataset ridotto a {len(df)} righe. Categorie uniche: {df['category'].nunique()}")

def document_vector(model, doc):
    words = gensim.utils.simple_preprocess(str(doc))
    word_vectors = []

    for word in words:
        if word in model.key_to_index:
            word_vectors.append(model[word])

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)

print("\n4. Vettorizzazione in corso con Word2Vec...")
X_w2v = np.array([document_vector(w2v_google_news_model, text) for text in df['headline']])

print("5. Vettorizzazione in corso con FastText...")
X_ft = np.array([document_vector(ft_wiki_news_model, text) for text in df['headline']])

y = df['category']
X_w2v_train, X_w2v_test, y_train, y_test = train_test_split(X_w2v, y, test_size=0.2, random_state=42)
X_ft_train, X_ft_test, _, _ = train_test_split(X_ft, y, test_size=0.2, random_state=42)

clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)

print("\n6. Addestramento classificatore con Word2Vec...")
clf.fit(X_w2v_train, y_train)
preds_w2v = clf.predict(X_w2v_test)
acc_w2v = accuracy_score(y_test, preds_w2v)
print(f"-> Accuratezza Word2Vec: {acc_w2v:.4f}")

print("\n7. Addestramento classificatore con FastText...")
clf.fit(X_ft_train, y_train)
preds_ft = clf.predict(X_ft_test)
acc_ft = accuracy_score(y_test, preds_ft)
print(f"-> Accuratezza FastText: {acc_ft:.4f}")

1. Ricaricamento Word2Vec (Google News)...
2. Ricaricamento FastText (Wiki+News)...

3. Caricamento del dataset...
Dataset ridotto a 10000 righe. Categorie uniche: 26

4. Vettorizzazione in corso con Word2Vec...
5. Vettorizzazione in corso con FastText...

6. Addestramento classificatore con Word2Vec...


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


-> Accuratezza Word2Vec: 0.6095

7. Addestramento classificatore con FastText...


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


-> Accuratezza FastText: 0.6565


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import gensim.utils
from gensim.models.doc2vec import Doc2Vec

# 1. Prepariamo i dati
df = pd.read_csv('news_headline_classification.csv').head(10000)
headlines = df['headline'].astype(str).tolist()
y = df['category']

# Rinizializziamo il classificatore
clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)

# --- 1. DOC2VEC (Text8) ---
print("Vettorizzazione in corso con Doc2Vec...")
try:
    d2v_model = Doc2Vec.load("doc2vec_text8.model")
    # infer_vector calcola il vettore per le nuove frasi
    X_d2v = np.array([d2v_model.infer_vector(gensim.utils.simple_preprocess(text)) for text in headlines])

    X_d2v_train, X_d2v_test, y_train, y_test = train_test_split(X_d2v, y, test_size=0.2, random_state=42)
    print("Addestramento classificatore con Doc2Vec...")
    clf.fit(X_d2v_train, y_train)
    print(f"-> Accuratezza Doc2Vec: {accuracy_score(y_test, clf.predict(X_d2v_test)):.4f}\n")
except FileNotFoundError:
    print("Errore: file 'doc2vec_text8.model' non trovato. Assicurati di aver eseguito la Question 6.")

# --- 2. INFERSENT ---
print("Vettorizzazione in corso con InferSent (potrebbe richiedere 1-2 minuti)...")
try:
    # InferSent ha bisogno di costruire il vocabolario sulle frasi prima di codificarle
    infersent.build_vocab(headlines, tokenize=True)
    X_infer = infersent.encode(headlines, tokenize=True)

    X_infer_train, X_infer_test, _, _ = train_test_split(X_infer, y, test_size=0.2, random_state=42)
    print("Addestramento classificatore con InferSent...")
    clf.fit(X_infer_train, y_train)
    print(f"-> Accuratezza InferSent: {accuracy_score(y_test, clf.predict(X_infer_test)):.4f}")
except NameError:
    print("Errore: 'infersent' non è definito. Assicurati di aver eseguito le celle della Question 8 in questa sessione.")

Vettorizzazione in corso con Doc2Vec...
Addestramento classificatore con Doc2Vec...


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


-> Accuratezza Doc2Vec: 0.4615

Vettorizzazione in corso con InferSent (potrebbe richiedere 1-2 minuti)...
Found 13646(/15243) words with w2v vectors
Vocab size : 13646
Addestramento classificatore con InferSent...
-> Accuratezza InferSent: 0.6930


**Word2Vec + Average aggregation function**

In [ ]:
# your code here

**FastText + Average aggregation function**

In [ ]:
# your code here

**Doc2Vec (Text8)**

In [ ]:
# your code here

**InferSent**

In [ ]:
# your code here